In [ ]:
# Residential proximity to oil and gas development and socioeconomic conditions: A comparative analysis of 33 countries
# 01a: PopExposure Prep

# This file preps data to apply Popexposure to finds the number of people living near buffered oil and gas wells. 
# Author: Lara Schwarz
# Last updated: November 2025
# Reviewed by Nina Flores September 2025

# Steps:

## prep step 1: Identifying countries with oil and gas wells within 3km of their borders
## Prep step 2: Create a parquet file in sharepoint for each country with ogd wells 
## Prep step 3: Create a parquet for each country with 3km buffer and only buffer (for outside of country estimates)
    # Prep step 3.1: exporting shapefile for all countries with 3km buffer
## Prep step 4: Create a oil and gas file for each country using 3km buffer
## Prep step 5: Identifying offshore wells using MODIS water mask
## Prep step 6: Identifying out of country wells including country names of wells outside of the country (using modis)
## Prep step 7: counting how many ogd wells in each country + 5km
## Prep step 8: counting how many ogd wells in each ADM2 unit



In [1]:
## Importing libraries

import geopandas as gpd
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from thefuzz import process
import pandas as pd
import pathlib
import pandas as pd
import sys
import os
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import geopandas as gpd
from rasterio.plot import show
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import fiona
import geopandas as gpd
from shapely.geometry import Polygon
import ee
import geemap
import matplotlib.cm as cm
import matplotlib.colors as colors
from shapely.geometry import box
import math
from shapely.geometry import Point
import glob
import seaborn as sns
import openpyxl
import subprocess
from pathlib import Path
from popexposure import PopEstimator



/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/geemap/conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# Defining base paths
repo = pathlib.Path.cwd().parent.parent
share_path = pathlib.Path("/Users/larasch/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/data")

# Define paths
oil_path = share_path / "environmental/oil_and_gas/raw_data/ogim.parquet"
shapefile_path = repo/ "data/OneDrive_1_12-11-2024/lbd_standard_admin_2.shp"

# Define the directory for the country-specific parquet files
country_dir = share_path / "geo_boundaries/processed_data/countries_ogd"

# Define the directory for the OGIM country-specific parquet files
ogim_dir = share_path / "environmental/oil_and_gas/raw_data/ogim_by_country"

# Define the directory for results
results_dir = repo / "results"

# List of countries with oil and gas (identified in prep step 1)
# Note: Nigeria is excluded because all oil and gas wells are more than 5km offshore
country_list = [
    "Argentina", "Australia", "Austria", "Belgium","Bolivia", "Brazil",
    "Canada", "Chile", "Colombia", "Denmark", "Ecuador", "Ethiopia",  "France", "Guatemala",
    "Germany", "Ireland", "Italy", "Libya", "Mexico", "Mozambique","Netherlands",   
    "New Zealand",  "Norway","Papua New Guinea","Paraguay","Peru", "Saudi Arabia",
    "South Sudan", "Sudan","United Arab Emirates",  "United Kingdom", "United States", "Venezuela"  
]


In [ ]:
# prep step 1: Identifying countries with oil and gas wells within 5km of their borders

# Load data
oil = gpd.read_parquet(oil_path)
global_shapefile = gpd.read_file(shapefile_path)
print(len(oil))
print(global_shapefile.head())

# Dissolve by ADM0_NAME to get one geometry per country
countries = global_shapefile.dissolve(by="ADM0_NAME", as_index=False)

# check result
print(countries.head())

# Reproject both to a projected CRS for buffering (EPSG:3395 - World Mercator, so units in meters)
oil_proj = oil.to_crs(epsg=3395)
countries_proj = countries.to_crs(epsg=3395)

# Buffer oil wells by 5km (used to be 3km, now corrected)
oil_buffered = oil_proj.copy()
oil_buffered['geometry'] = oil_proj.buffer(5000)

# Save the buffered oil wells
#oil_buffered.to_parquet(output_path)
#print(f" Saved buffered oil wells to {output_path}")

# Spatial join: find countries intersecting buffered oil wells
intersecting = gpd.sjoin(countries_proj, oil_buffered, how="inner", predicate="intersects")
total_wells_intersecting = intersecting['index_right'].nunique()
print(f"Total number of oil wells intersecting countries: {total_wells_intersecting}")

# Get unique country names
countries_with_oil_buffers = sorted(intersecting["ADM0_NAME"].unique())

# Print results
print("\n Countries with buffered oil wells intersecting:")
for country in countries_with_oil_buffers:
    print(country)

# Plot to visualize
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
countries.to_crs(epsg=4326).plot(ax=ax, color='lightgray', edgecolor='black')
oil_buffered.to_crs(epsg=4326).plot(ax=ax, color='orange', alpha=0.5, label='Buffered Oil Wells')
plt.title("Countries Intersecting 5km Buffered Oil Wells")
plt.legend()
plt.tight_layout()
plt.show()

4519663
      geo_id  ad2_id  ad0_parent  ad1_parent  ADM2_CODE       ADM2_NAME  \
0  1001143.0       1         143           1    1001143  Aguascalientes   
1  2001143.0       2         143           1    2001143        Asientos   
2  3001143.0       3         143           1    3001143        Calvillo   
3  4001143.0       4         143           1    4001143           Cosio   
4  5001143.0       5         143           1    5001143        El Llano   

   ADM1_CODE       ADM1_NAME  ADM0_CODE ADM0_NAME  loc_id  \
0       1143  Aguascalientes        143    Mexico   50560   
1       1143  Aguascalientes        143    Mexico   50561   
2       1143  Aguascalientes        143    Mexico   50562   
3       1143  Aguascalientes        143    Mexico   50563   
4       1143  Aguascalientes        143    Mexico   50564   

                                            geometry  
0  POLYGON ((-102.09772 22.0232, -102.0978 22.022...  
1  POLYGON ((-101.99941 22.21928, -101.99917 22.2...  
2  POLYGO

KeyboardInterrupt: 

In [ ]:
# Prep step 2: Create a parquet file in sharepoint for each country with ogd wells 


# Define the output parquet files for all countries
countries_path = country_dir / "countries.parquet"

# Load the countries data
global_shapefile = gpd.read_file(shapefile_path)

# Export each country as a separate Parquet file (not dissolved)
for country in country_list:
    country_df = global_shapefile[global_shapefile['ADM0_NAME'] == country].copy()
    country_df["geometry"] = country_df["geometry"].make_valid()
    country_df["ID_admin_unit"] = country_df["ADM2_CODE"]
    if not country_df.empty:
        output_path = country_dir / f"{country.replace(' ', '_').lower()}_adm2.parquet"
        country_df.to_parquet(output_path)
        print(f"Saved : {output_path}")
    else:
        print(f"No data found for {country}")

# Dissolve by ADM0_NAME to get one geometry per country
countries = global_shapefile.dissolve(by="ADM0_NAME", as_index=False)
countries.crs

# Save the dissolved GeoDataFrame
#countries.to_parquet(countries_path)

# Iterate over the country list and save a parquet file for each
for country in country_list:
    country_df = countries[countries['ADM0_NAME'] == country]
    print(country_df.head())
    country_df["geometry"]=country_df["geometry"].make_valid()
    country_df["ID_admin_unit"] = country_df["ADM0_NAME"].str.lower().str.replace(" ", "_")

    if not country_df.empty:
        output_path = country_dir / f"{country.replace(' ', '_').lower()}.parquet"
        country_df.to_parquet(output_path)
        print(f"Saved {output_path}")
    else:
        print(f"No data found for {country}")

In [ ]:
# Prep step 3: Create a parquet for each country with 5km buffer and only buffer (for outside of country estimates)

# Initialize GEE
ee.Authenticate()
ee.Initialize()

# Load the countries data
countries = gpd.read_parquet(countries_path)

# Buffer distance in meters
buffer_distance = 5000

# Function to get the appropriate UTM CRS for a given geometry (this could be edited to be even more specific)
def get_utm_crs(geometry):
    """Get the appropriate UTM CRS (north or south) for the centroid of a geometry."""
    centroid = geometry.centroid
    lon = centroid.x
    lat = centroid.y
    utm_zone = int((lon + 180) / 6) + 1
    if lat >= 0:
        epsg_code = 32600 + utm_zone  # Northern hemisphere
    else:
        epsg_code = 32700 + utm_zone  # Southern hemisphere
    return f"EPSG:{epsg_code}"

# Loop through each country and create 3km buffer
for country_name in country_list:
    country_gdf = countries[countries["ADM0_NAME"] == country_name]
    
    if not country_gdf.empty:
       # Ensure geometries are valid
        country_gdf["geometry"] = country_gdf["geometry"].make_valid()
        country_gdf["geometry"] = country_gdf["geometry"].apply(lambda x: x if x.is_valid else x.buffer(0)) 
        
        # Dynamically determine UTM CRS
        utm_crs = get_utm_crs(country_gdf.geometry.union_all())
        print(f"Using CRS {utm_crs} for {country_name}")
        
        # Project to UTM CRS
        country_gdf = country_gdf.to_crs(utm_crs)  

        #country_gdf.plot()

        # Special case for Chile: apply a 0 buffer to fix geometry issues
        if country_name == "Chile":
            country_gdf["geometry"] = country_gdf["geometry"].buffer(0)
            country_gdf["geometry"] = country_gdf["geometry"].simplify(100) 
        
        country_gdf["geometry"] = country_gdf["geometry"].make_valid()
        
        # Create a 3km buffer
        country_gdf["geometry_buffered"] = country_gdf["geometry"].buffer(buffer_distance)
        country_gdf["ID_admin_unit"] = country_gdf["ADM0_NAME"].str.lower().str.replace(" ", "_")

        # Save the buffered country polygon
        buffered_gdf = country_gdf.copy()
        buffered_gdf = buffered_gdf[["ADM0_NAME", "geometry_buffered"]].rename(columns={"geometry_buffered": "geometry"})
        #buffered_path = country_dir / f"{country_name.replace(' ', '_').lower()}_3km_buffer.parquet"
        buffered_path = country_dir / f"{country_name.replace(' ', '_').lower()}_5km_buffer.parquet" # updated to 5km
        buffered_gdf.to_parquet(buffered_path)

        # Create only the 3km buffer (excluding the country)
        country_buffer = buffered_gdf.copy()
        country_buffer["geometry"] = country_buffer["geometry"].difference(country_gdf["geometry"])
        #buffer_only_path = country_dir / f"{country_name.replace(' ', '_').lower()}_only_3km_buffer.parquet"
        buffer_only_path = country_dir / f"{country_name.replace(' ', '_').lower()}_only_5km_buffer.parquet" # updated to 5km
        country_buffer.to_parquet(buffer_only_path)

        print(f"Saved {buffered_path} and {buffer_only_path}")


In [ ]:
# Prep step 3.1: exporting shapefile for all countries with 5km buffer

all_buffers = []
target_crs = "EPSG:4326"

for country_name in country_list:
    buffered_path = country_dir / f"{country_name.replace(' ', '_').lower()}_5km_buffer.parquet"
    if buffered_path.exists():
        gdf = gpd.read_parquet(buffered_path)
        gdf["country"] = country_name
        if target_crs is None:
            target_crs = gdf.crs  # Use the CRS of the first file
        if gdf.crs != target_crs:
            gdf = gdf.to_crs(target_crs)
        all_buffers.append(gdf)
    else:
        print(f"Buffer file not found for {country_name}")

# Combine all buffers into one GeoDataFrame
if all_buffers:
    combined_gdf = gpd.GeoDataFrame(pd.concat(all_buffers, ignore_index=True), crs=all_buffers[0].crs)
    # Export to geoJSON
    # Dissolve all geometries into one (single outer polygon)
    outer_polygon = combined_gdf.unary_union

    # Convert back to a GeoDataFrame for export
    outer_gdf = gpd.GeoDataFrame(geometry=[outer_polygon], crs=combined_gdf.crs)

    # Export as GeoJSON
    outer_gdf.to_file(country_dir / "all_countries_5km_buffer_outer.geojson", driver="GeoJSON")
    combined_gdf.to_file(country_dir / "all_countries_5km_buffer.geojson", driver="GeoJSON")
    print("Combined shapefile saved!")
else:
    print("No buffer files found.")


fig, ax = plt.subplots(figsize=(12, 8))
combined_gdf.plot(ax=ax, column="country", legend=True, edgecolor='black')
plt.title("Combined 5km Buffers for All Countries")
plt.show()

In [ ]:
# Prep step 4: Create a oil and gas file for each country using 5km buffer

# Load the oil data
oil_gdf = gpd.read_parquet(oil_path)


# Loop through each country and save the oil data within the 5km buffer
for country in country_list:
    print(f"Processing {country}...")
    buffer_path = country_dir / f"{country.lower().replace(' ', '_')}_5km_buffer.parquet"
    
    if buffer_path.exists():
        # Read the 3km buffer
        buffer_gdf = gpd.read_parquet(buffer_path)
        buffer_gdf_crs = buffer_gdf.crs
        oil_gdf_v2= oil_gdf.to_crs(buffer_gdf_crs)
        print(len(oil_gdf_v2))
        # Spatial join to find oil points within the buffer
        oil_in_buffer = gpd.sjoin(oil_gdf_v2, buffer_gdf, predicate="intersects",how="inner")
        
        # Update buffer distance
        oil_in_buffer["buffer_dist_1km"] = 1000 # buffer distance in in meters 
        oil_in_buffer["buffer_dist_3km"] = 3000 # buffer distance in in meters 
        oil_in_buffer["buffer_dist_5km"] = 5000 # buffer distance in in meters
        oil_in_buffer["ID_hazard"] = range(1, len(oil_in_buffer) + 1)
        oil_in_buffer["ID_hazard"] = oil_in_buffer["ID_hazard"].astype(str)

        #oil_in_buffer.rename(columns={"id": "ID_climate_hazard"}, inplace=True)
        oil_in_buffer = oil_in_buffer.drop(columns=["index_right"])
        print(oil_in_buffer.info())  
    
        if not oil_in_buffer.empty:
            # Save the filtered oil data to parquet
            output_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}.parquet"
            oil_in_buffer = oil_in_buffer.to_crs(buffer_gdf_crs)
            oil_in_buffer.to_parquet(output_path)
            print(f"Saved {output_path}")
        else:
            print(f"No oil data found in 5km buffer for {country}")
    else:
        print(f"Buffer file for {country} not found.")






In [ ]:
# Prep step 5: Identifying offshore wells using MODIS water mask

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Summary stats placeholder
summary_stats = []

# Load MODIS water mask (single mosaic)
modis = ee.ImageCollection("MODIS/006/MOD44W") \
    .filter(ee.Filter.calendarRange(2015, 2015, 'year')) \
    .select("water_mask") \
    .mosaic() \
    .reproject(crs='EPSG:4326', scale=250)

water_crs = modis.projection().crs().getInfo()

chunk_size = 300  # Need to split into chunks to avoid memory issues

for country in country_list:
    print(f"\nProcessing {country}...")

    oil_path = ogim_dir / f"OGIM_{country.replace(' ', '_').lower()}.parquet"
    if not oil_path.exists():
        print(f"  No data found for {country}, skipping.")
        continue

    # Read wells with geopandas
    oil_gdf = gpd.read_parquet(oil_path)

    # Clean geometries: only valid points
    oil_gdf = oil_gdf[oil_gdf.geometry.notnull()]
    oil_gdf = oil_gdf[oil_gdf.geometry.type == "Point"]
    oil_gdf = oil_gdf[~oil_gdf.geometry.is_empty]

    # Reproject to match MODIS water mask CRS if needed
    if oil_gdf.crs != water_crs:
        oil_gdf = oil_gdf.to_crs(water_crs)

    print(f"  Total wells before offshore check: {len(oil_gdf)}")

    offshore_gdfs = []
    total_wells = len(oil_gdf)

    for i in range(0, total_wells, chunk_size):
        print(f" - Processing chunk {i // chunk_size + 1}")
        oil_chunk = oil_gdf.iloc[i:i + chunk_size]

        # Convert to EE Features
        def row_to_ee_feature(row):
            coords = [row.geometry.x, row.geometry.y]
            return ee.Feature(ee.Geometry.Point(coords)).set("index", row.name)

        features = oil_chunk.apply(row_to_ee_feature, axis=1).tolist()
        fc = ee.FeatureCollection(features)

        # Use reduceRegions to sample water_mask for all wells in the chunk
        sampled = modis.reduceRegions(
            collection=fc,
            reducer=ee.Reducer.first(),
            scale=250,
            tileScale=4
        )

        # Filter wells that overlap water (water_mask == 1)
        offshore_fc = sampled.filter(ee.Filter.eq("first", 1))

        # Get list of indexes of offshore wells
        try:
            offshore_list = offshore_fc.aggregate_array("index").getInfo()
            offshore_gdf = oil_chunk.loc[offshore_list].copy()
            if not offshore_gdf.empty:
                offshore_gdfs.append(offshore_gdf)
        except Exception as e:
            print(f"   - Error in chunk {i // chunk_size + 1}: {e}")

    # Combine and save results
    if offshore_gdfs:
        offshore_combined = pd.concat(offshore_gdfs, ignore_index=True)
        offshore_combined["ID_climate_hazard"] = range(1, len(offshore_combined) + 1)
        offshore_combined["ID_climate_hazard"] = offshore_combined["ID_climate_hazard"].astype(str)

        # Update buffer distance
        offshore_combined["buffer_dist_1km"] = 1000 # buffer distance in in meters 
        offshore_combined["buffer_dist_3km"] = 3000 # buffer distance in in meters 
        offshore_combined["buffer_dist_5km"] = 5000 # buffer distance in in meters
        offshore_combined["ID_hazard"] = range(1, len(offshore_combined) + 1)
        offshore_combined["ID_hazard"] = offshore_combined["ID_hazard"].astype(str)

        # Save
        output_path = ogim_dir / f"OGIM_{country.replace(' ', '_').lower()}_offshore_modis.parquet"
        offshore_combined.to_parquet(output_path)
        print(f"  Offshore wells saved to {output_path}")

        offshore_wells = len(offshore_combined)
    else:
        print("  No offshore wells found.")
        offshore_wells = 0

    print(f"Total wells: {total_wells}")
    print(f"Offshore wells: {offshore_wells}")

    summary_stats.append({
        "country": country,
        "total_wells": total_wells,
        "offshore_wells": offshore_wells
    })
    
    

In [ ]:
# Prep step 6: Identifying out of country wells including country names of wells outside of the country (using modis)
# Selecting out of country wells

# Initialize Earth Engine
ee.Initialize()

# Load global shapefile and dissolve to country polygons
global_shapefile = gpd.read_file(shapefile_path)
countries = global_shapefile.dissolve(by="ADM0_NAME", as_index=False)

# Load MODIS water mask (single mosaic)
modis = ee.ImageCollection("MODIS/006/MOD44W") \
    .filter(ee.Filter.calendarRange(2015, 2015, 'year')) \
    .select("water_mask") \
    .mosaic() \
    .reproject(crs='EPSG:4326', scale=250)
water_proj = modis.projection().crs().getInfo()

def flag_offshore_modis(feature):
    sampled = modis.sample(
        region=feature.geometry(),
        scale=250,
        numPixels=1,
        geometries=True
    ).first()
    water_value = ee.Algorithms.If(
        sampled,
        ee.Number(sampled.get('water_mask')),
        0
    )
    is_water = ee.Algorithms.If(
        ee.Number(water_value).eq(1),
        1,
        0
    )
    return feature.set('on_water', is_water)

summary_stats = []

for country in country_list:
    print(f"Processing {country}...")

    boundary_path = country_dir / f"{country.lower().replace(' ', '_')}_only_5km_buffer.parquet"
    oil_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}.parquet"
    
    if not boundary_path.exists():
        print(f"Buffer file for {country} not found — skipping.")
        continue
    if not oil_path.exists():
        print(f"Oil data for {country} not found — skipping.")
        continue

    # Load buffer and oil wells
    country_buffer = gpd.read_parquet(boundary_path)
    oil_gdf = gpd.read_parquet(oil_path)
    total_wells = len(oil_gdf)

    # Reproject if needed
    if oil_gdf.crs != country_buffer.crs:
        oil_gdf = oil_gdf.to_crs(country_buffer.crs)
    if country_buffer.crs != water_proj:
        country_buffer = country_buffer.to_crs(water_proj)
        oil_gdf = oil_gdf.to_crs(water_proj)

    # Drop old index_right if it exists
    if 'index_right' in oil_gdf.columns:
        oil_gdf = oil_gdf.drop(columns=['index_right'])
    
    # Select wells inside 3km buffer
    oil_in_buffer = gpd.sjoin(oil_gdf, country_buffer, predicate="intersects", how="inner")
    print(f"Wells inside buffer: {len(oil_in_buffer)}")

    if oil_in_buffer.empty:
        print(f"No wells inside 5km buffer for {country} — skipping.")
        continue

    # Convert wells to Earth Engine FeatureCollection
    ee_points = [ee.Feature(ee.Geometry.Point(xy)) for xy in zip(oil_in_buffer.geometry.x, oil_in_buffer.geometry.y)]
    oil_fc = ee.FeatureCollection(ee_points)

    # Flag points based on MODIS water
    oil_fc_flagged = oil_fc.map(flag_offshore_modis)

    # Split into offshore (on_water == 1) and onshore (on_water == 0)
    wells_on_land = oil_fc_flagged.filter(ee.Filter.eq('on_water', 0))
    wells_on_water = oil_fc_flagged.filter(ee.Filter.eq('on_water', 1))

    offshore_wells = wells_on_water.size().getInfo()
    onshore_wells = wells_on_land.size().getInfo()

    summary_stats.append({
        "country": country,
        "total_wells": total_wells,
        "wells_in_buffer": len(oil_in_buffer),
        "onshore_wells": onshore_wells,
        "offshore_wells": offshore_wells
    })

    if wells_on_land.size().getInfo() == 0:
        print(f"No out of country wells for {country} — skipping saving.")
        continue

    # Convert onshore wells back to GeoDataFrame
    land_gdf = geemap.ee_to_gdf(wells_on_land)

    if land_gdf.empty:
        print(f"No out of country wells found after conversion for {country} — skipping saving.")
        continue

    # Ensure CRS matches
    if countries.crs != land_gdf.crs:
        countries = countries.to_crs(land_gdf.crs)

    # Spatial join to assign country name
    land_gdf = gpd.sjoin(land_gdf, countries[["ADM0_NAME", "geometry"]], predicate="intersects", how="inner")
    land_gdf = land_gdf.rename(columns={"ADM0_NAME": "well_country"}).drop(columns=["index_right"])

    # Assign unique ID
    land_gdf["ID_climate_hazard"] = land_gdf["well_country"]

    land_gdf["buffer_dist_1km"] = 1000 # buffer distance in in meters 
    land_gdf["buffer_dist_3km"] = 3000 # buffer distance in in meters 
    land_gdf["buffer_dist_5km"] = 5000 # buffer distance in in meters
    land_gdf["ID_hazard"] = range(1, len(land_gdf) + 1)
    land_gdf["ID_hazard"] = land_gdf["ID_hazard"].astype(str)
    
    #  Summarize number of wells per country ---
    country_counts = land_gdf["well_country"].value_counts().reset_index()
    country_counts.columns = ["well_country", "num_out_of_country_wells"]
    print(f"Out-of-country wells for {country} by country:\n", country_counts)

    # Optionally, add this summary to your summary_stats
    for _, row in country_counts.iterrows():
        summary_stats.append({
            "origin_country": country,
            "well_country": row["well_country"],
            "num_out_of_country_wells": row["num_out_of_country_wells"]
        })

    # Save
    output_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}_out_of_country_modis_v2.parquet"
    land_gdf.to_parquet(output_path)
    print(f"Saved {len(land_gdf)} land wells for {country}: {output_path}")

# Create summary table
summary_df = pd.DataFrame(summary_stats)
print(summary_df)

In [ ]:
## Prep step 7: counting how many ogd wells in each country + 5km

well_counts = []
total_rows = 0

for country in country_list:
    row = {"country": country}
    # Total wells
    total_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}_v2.parquet"
    if total_path.exists():
        df = gpd.read_parquet(total_path)
        n_total = len(df)
        total_rows += n_total
        print(f"{country}: {n_total} total wells")
    else:
        n_total = 0
        print(f"{country}: Total wells file not found.")
    row["total_wells"] = n_total

    # Offshore wells
    offshore_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}_offshore_modis_v2.parquet"
    if offshore_path.exists():
        n_offshore = len(gpd.read_parquet(offshore_path))
        print(f"    {n_offshore} offshore wells")
    else:
        n_offshore = 0
        print(f"    Offshore wells file not found.")
    row["offshore_wells"] = n_offshore

    # Out of country wells
    outcountry_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}_out_of_country_modis_v2.parquet"
    if outcountry_path.exists():
        n_outcountry = len(gpd.read_parquet(outcountry_path))
        print(f"    {n_outcountry} out of country wells")
    else:
        n_outcountry = 0
        print(f"    Out of country wells file not found.")
    row["out_of_country_wells"] = n_outcountry

    well_counts.append(row)

# Create summary table
well_counts_df = pd.DataFrame(well_counts)
print("\nSummary table:")
print(well_counts_df)

# Optionally, save to CSV
well_counts_df.to_csv(results_dir / "well_counts_summary.csv", index=False)

# Final total
print(f"\n Total rows across all files: {total_rows}")

In [ ]:
## Prep step 8: counting how many ogd wells in each ADM2 unit

adm2_counts = []

for country in country_list:
    wells_path = ogim_dir / f"OGIM_{country.lower().replace(' ', '_')}_v2.parquet"
    adm2_path = country_dir / f"{country.lower().replace(' ', '_')}_adm2.parquet"
    if wells_path.exists() and adm2_path.exists():
        wells_gdf = gpd.read_parquet(wells_path)
        adm2_gdf = gpd.read_parquet(adm2_path)
        # Ensure CRS matches
        if wells_gdf.crs != adm2_gdf.crs:
            wells_gdf = wells_gdf.to_crs(adm2_gdf.crs)
        # Spatial join: assign each well to an ADM2 unit
        joined = gpd.sjoin(wells_gdf, adm2_gdf[['ADM2_CODE', 'geometry']], predicate="intersects", how="inner")
        # Count wells per ADM2
        counts = joined.groupby('ADM2_CODE').size().reset_index(name='well_count')
        counts['country'] = country
        adm2_counts.append(counts)
        print(f"{country}: counted wells in {len(counts)} ADM2 units")
    else:
        print(f"{country}: missing well or ADM2 file.")

# Combine all countries
if adm2_counts:
    adm2_counts_df = pd.concat(adm2_counts, ignore_index=True)
    print(adm2_counts_df.head())
    # Save to CSV
    adm2_counts_df.to_csv(results_dir / "well_counts_by_adm2.csv", index=False)
    print(f" Saved: {results_dir / 'well_counts_by_adm2.csv'}")
else:
    print("No ADM2 well counts calculated.")